# Subastas Casanare

Version: 1.9.2

Datos en Google Drive: **`MyDrive/subasta_ganadera_data/`**

## Uso en Colab

1. **Celda A** — monta Drive, funciones y precarga.
2. **Celda B** — panel (Descarga · Datos · Exportar).
3. **Celda C** — graficos (Plotly + tabla).
4. **Celda D** — analisis ML (EDA para precio maximo de compra).


In [ ]:
# Celda A - Configuracion (montar Drive + funciones)
!pip install -q pandas pdfplumber requests pyarrow plotly ipywidgets kaleido

import re
from pathlib import Path

import pandas as pd
import pdfplumber
import requests
from google.colab import drive

drive.mount("/content/drive")

PROYECTO = Path("/content/drive/MyDrive/subasta_ganadera_data")
LOTE_DIR = PROYECTO / "data" / "lotes"
RESUMEN_DIR = PROYECTO / "data" / "resumen"
COMBINED_DIR = PROYECTO / "data" / "combined"
EXPORT_DIR = PROYECTO / "export"
PDF_DIR = PROYECTO / "pdfs"
for d in [LOTE_DIR, RESUMEN_DIR, COMBINED_DIR, EXPORT_DIR, PDF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ROW_PATTERN = re.compile(
    r"^(\d+)\s+([A-Z]{2})\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+"
    r"(.+?)\s+(\d{2}:\d{2}:\d{2})\s+([\d.]+)\s+([\d.]+)(?:\s+(.*))?$"
)
SUMMARY_ROW_PATTERN = re.compile(
    r"^(.+?)\s+([A-Z]{2})\s+(\d+)\s+"
    r"([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)$"
)
HEADER_PATTERN = re.compile(
    r"FERIA NO\.\s*(\d+).*?CIUDAD\s+(.+?)\s+FECHA FERIA\.\s*(\d{4}-\d{2}-\d{2})", re.DOTALL
)
SECTION_PATTERN = re.compile(r"^(REMATE|SUBASTA)\s+(.+)$")
LOTES_SKIP = ("SUBASTA", "LISTADO", "FERIA", "FECHA", "Lote Sexo", "RESULTADOS", "TOTALES:", "REMATE")
RESUMEN_SKIP = ("SUBASTA GANADERA", "LISTADO", "FERIA", "FECHA", "Lote Sexo", "RESULTADOS GENERALES", "Edad Sexo", "TOTALES:")


def parse_num(v):
    return int(str(v).replace(".", ""))


def listar_ferias_en_drive():
    return sorted(int(p.stem.replace("feria_", "")) for p in LOTE_DIR.glob("feria_*.parquet"))


def ultima_feria_en_drive():
    ferias = listar_ferias_en_drive()
    return ferias[-1] if ferias else 0


def download_pdf(feria: int) -> Path:
    url = f"https://www.subacasanare.com/Precio_Pdf/{feria}"
    dest = PDF_DIR / f"{feria}.pdf"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        raise ValueError("respuesta no es PDF")
    dest.write_bytes(r.content)
    return dest


def extract_metadata(text: str) -> dict:
    m = HEADER_PATTERN.search(text)
    if not m:
        return {}
    return {"feria_no": m.group(1), "ciudad": m.group(2).strip(), "fecha_subasta": m.group(3)}


def extract_lotes(pdf_path: Path) -> pd.DataFrame:
    records, current, in_summary = [], None, False
    with pdfplumber.open(pdf_path) as pdf:
        meta = extract_metadata(pdf.pages[0].extract_text() or "")
        for page_num, page in enumerate(pdf.pages, 1):
            for line in (page.extract_text() or "").split("\n"):
                line = line.strip()
                if "RESULTADOS GENERALES" in line:
                    in_summary = True
                if not line or in_summary or line.startswith(LOTES_SKIP):
                    continue
                match = ROW_PATTERN.match(line)
                if match:
                    if current:
                        records.append(current)
                    current = {
                        "pagina": page_num, "lote": int(match.group(1)), "sexo": match.group(2),
                        "cantidad": int(match.group(3)), "peso_total": parse_num(match.group(4)),
                        "peso_promedio": parse_num(match.group(5)), "procedencia": match.group(6).strip(),
                        "entrada": match.group(7), "precio_base": parse_num(match.group(8)),
                        "precio_final": parse_num(match.group(9)), "observaciones": (match.group(10) or "").strip(),
                    }
                elif current:
                    current["observaciones"] = (current["observaciones"] + " " + line).strip()
    if current:
        records.append(current)
    df = pd.DataFrame(records)
    for k, v in meta.items():
        df[k] = v
    return df


def extract_resumen(pdf_path: Path) -> pd.DataFrame:
    records, current_section, in_summary = [], None, False
    with pdfplumber.open(pdf_path) as pdf:
        meta = extract_metadata(pdf.pages[0].extract_text() or "")
        for page_num, page in enumerate(pdf.pages, 1):
            for line in (page.extract_text() or "").split("\n"):
                line = line.strip()
                if not line:
                    continue
                if "RESULTADOS GENERALES" in line:
                    in_summary = True
                    continue
                if not in_summary or line.startswith(RESUMEN_SKIP):
                    continue
                sec = SECTION_PATTERN.match(line)
                if sec:
                    current_section = f"{sec.group(1)} {sec.group(2).strip()}"
                    continue
                match = SUMMARY_ROW_PATTERN.match(line)
                if match:
                    records.append({
                        "pagina": page_num, "seccion": current_section,
                        "edad": match.group(1).strip(), "sexo": match.group(2),
                        "cantidad": int(match.group(3)), "peso_med": parse_num(match.group(4)),
                        "minimo": parse_num(match.group(5)), "maximo": parse_num(match.group(6)),
                        "promedio": parse_num(match.group(7)), "vr_animal": parse_num(match.group(8)),
                    })
    df = pd.DataFrame(records)
    for k, v in meta.items():
        df[k] = v
    return df


def feria_procesada(feria: int) -> bool:
    p = LOTE_DIR / f"feria_{feria}.parquet"
    if not p.exists() or p.stat().st_size == 0:
        return False
    try:
        return not pd.read_parquet(p).empty
    except Exception:
        return False


def guardar_feria(lotes_df: pd.DataFrame, resumen_df: pd.DataFrame, feria: int):
    lotes_df = lotes_df.copy()
    lotes_df["fecha_subasta"] = pd.to_datetime(lotes_df["fecha_subasta"], errors="coerce")
    lotes_df.to_parquet(LOTE_DIR / f"feria_{feria}.parquet", index=False)
    if not resumen_df.empty:
        resumen_df = resumen_df.copy()
        resumen_df["fecha_subasta"] = pd.to_datetime(resumen_df["fecha_subasta"], errors="coerce")
        resumen_df.to_parquet(RESUMEN_DIR / f"feria_{feria}.parquet", index=False)


def rebuild_combined():
    lotes_files = sorted(LOTE_DIR.glob("feria_*.parquet"))
    resumen_files = sorted(RESUMEN_DIR.glob("feria_*.parquet"))
    lotes = pd.concat([pd.read_parquet(f) for f in lotes_files], ignore_index=True) if lotes_files else pd.DataFrame()
    resumen = pd.concat([pd.read_parquet(f) for f in resumen_files], ignore_index=True) if resumen_files else pd.DataFrame()
    if not lotes.empty:
        lotes.to_parquet(COMBINED_DIR / "lotes.parquet", index=False)
    if not resumen.empty:
        resumen.to_parquet(COMBINED_DIR / "resumen.parquet", index=False)
    return lotes, resumen


def _procesar_feria(feria: int, force: bool = False):
    if not force and feria_procesada(feria):
        return "skip", feria, None
    pdf = download_pdf(feria)
    lotes_df = extract_lotes(pdf)
    resumen_df = extract_resumen(pdf)
    if lotes_df.empty:
        raise ValueError("sin lotes")
    guardar_feria(lotes_df, resumen_df, feria)
    return "ok", feria, (len(lotes_df), len(resumen_df))


def run_single_feria(feria: int, force: bool = False):
    if not force and feria_procesada(feria):
        print(f"[SKIP] {feria} ya procesada")
        return "skip", feria
    status, num, counts = _procesar_feria(feria, force=force)
    rebuild_combined()
    precargar_datos(rebuild=False, verbose=False)
    print(f"[OK] {num} - {counts[0]} lotes, {counts[1]} resumen")
    return status, num


def run_batch(desde: int, hasta: int, force: bool = False, progress_cb=None):
    if desde > hasta:
        raise ValueError("'Desde' debe ser menor o igual que 'Hasta'")
    ok = skip = fail = 0
    total = hasta - desde + 1
    for i, feria in enumerate(range(desde, hasta + 1), start=1):
        if progress_cb:
            progress_cb(int(i / total * 100), feria)
        if not force and feria_procesada(feria):
            print(f"[SKIP] {feria}")
            skip += 1
            continue
        try:
            _, num, counts = _procesar_feria(feria, force=force)
            print(f"[OK] {num} - {counts[0]} lotes, {counts[1]} resumen")
            ok += 1
        except Exception as e:
            print(f"[FAIL] {feria}: {e}")
            fail += 1
    if progress_cb:
        progress_cb(100, None)
    lotes, resumen = rebuild_combined()
    precargar_datos(rebuild=False, verbose=False)
    print(f"\nResultado: OK={ok} SKIP={skip} FAIL={fail}")
    print(f"Combined: {len(lotes)} lotes | {len(resumen)} filas resumen")
    print(f"Guardado en: {PROYECTO}")
    return ok, skip, fail


CACHE = {"lotes": None, "resumen": None, "summary": None, "cargado": False}


def load_combined_lotes():
    if CACHE.get("cargado") and CACHE.get("lotes") is not None:
        return CACHE["lotes"]
    path = COMBINED_DIR / "lotes.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return rebuild_combined()[0]


def _stats_from_lotes(lotes, ferias):
    info = {
        "ferias": len(ferias),
        "ultima": ferias[-1] if ferias else None,
        "primera": ferias[0] if ferias else None,
        "lotes_total": len(lotes),
        "resumen_ferias": len(list(RESUMEN_DIR.glob("feria_*.parquet"))),
        "fecha_min": None,
        "fecha_max": None,
        "ciudades": [],
    }
    if not lotes.empty and "fecha_subasta" in lotes.columns:
        fechas = pd.to_datetime(lotes["fecha_subasta"], errors="coerce").dropna()
        if not fechas.empty:
            info["fecha_min"] = fechas.min().strftime("%Y-%m-%d")
            info["fecha_max"] = fechas.max().strftime("%Y-%m-%d")
    if not lotes.empty and "ciudad" in lotes.columns:
        info["ciudades"] = sorted(lotes["ciudad"].dropna().unique().tolist())
    return info


def precargar_datos(rebuild=False, verbose=True):
    """Lee Parquet desde Drive a memoria para graficos, tablas y export."""
    ferias = listar_ferias_en_drive()
    if not ferias:
        CACHE.update({"lotes": pd.DataFrame(), "resumen": pd.DataFrame(), "summary": None, "cargado": True})
        msg = "No hay ferias en Drive. Usa la pestaña Descarga."
        if verbose:
            print(msg)
        return {"ok": False, "mensaje": msg, "stats": _stats_from_lotes(pd.DataFrame(), [])}
    combined_path = COMBINED_DIR / "lotes.parquet"
    if rebuild or not combined_path.exists():
        if verbose:
            print("Regenerando consolidado desde ferias en Drive...")
        lotes, resumen = rebuild_combined()
    else:
        if verbose:
            print("Leyendo consolidado desde Drive...")
        lotes = pd.read_parquet(combined_path)
        resumen_path = COMBINED_DIR / "resumen.parquet"
        resumen = pd.read_parquet(resumen_path) if resumen_path.exists() else pd.DataFrame()
    CACHE["lotes"] = lotes
    CACHE["resumen"] = resumen
    CACHE["summary"] = build_price_summary(lotes) if not lotes.empty else None
    CACHE["cargado"] = True
    stats = _stats_from_lotes(lotes, ferias)
    msg = f"Precargado: {stats['lotes_total']:,} lotes | {stats['ferias']} ferias | {stats['fecha_min'] or '?'} → {stats['fecha_max'] or '?'}"
    if verbose:
        print(msg)
    return {"ok": True, "mensaje": msg, "stats": stats}


def datos_precargados():
    return CACHE["cargado"] and CACHE["lotes"] is not None and not CACHE["lotes"].empty


def load_feria_lotes(feria: int):
    path = LOTE_DIR / f"feria_{feria}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"No existe feria_{feria}.parquet")
    return pd.read_parquet(path)


def stats_drive():
    ferias = listar_ferias_en_drive()
    lotes = load_combined_lotes()
    return _stats_from_lotes(lotes, ferias)


MESES = {1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr", 5: "May", 6: "Jun",
         7: "Jul", 8: "Ago", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic"}

# --- Escala algoritmica: outliers y rango de ejes ---
SEXOS_ESPECIALES = frozenset({"EQ", "TP", "NP", "HV", "TO", "XX"})
OBS_OUTLIER_RE = re.compile(
    r"PRE[ÑN]AD|PRENAD|\d+\s*DIAS|EQUINO|CABALLO|EMBRION|SEMEN",
    re.IGNORECASE,
)
PRECIO_TOPE_GANADERO = 200_000   # COP: sobre esto = outlier para ganado corriente
IQR_OUTLIER_FACTOR = 1.5


def marcar_outliers_lotes(df: pd.DataFrame) -> pd.DataFrame:
    """Marca lotes atipicos (sexo especial, obs prenadas, IQR por sexo, tope absoluto)."""
    out = df.copy()
    out["precio_final"] = pd.to_numeric(out["precio_final"], errors="coerce")
    obs = out.get("observaciones", pd.Series("", index=out.index)).fillna("").astype(str)
    out["es_outlier"] = False
    out["motivo_outlier"] = ""

    m = out["sexo"].isin(SEXOS_ESPECIALES)
    out.loc[m, ["es_outlier", "motivo_outlier"]] = [True, "sexo_especial"]

    m = obs.str.contains(OBS_OUTLIER_RE, na=False) & ~out["es_outlier"]
    out.loc[m, ["es_outlier", "motivo_outlier"]] = [True, "obs_especial"]

    m = (out["precio_final"] > PRECIO_TOPE_GANADERO) & ~out["es_outlier"]
    out.loc[m, ["es_outlier", "motivo_outlier"]] = [True, "precio_tope"]

    base = out[~out["es_outlier"] & out["precio_final"].notna()].copy()
    if not base.empty:
        stats = base.groupby("sexo")["precio_final"].quantile([0.25, 0.75]).unstack()
        stats.columns = ["q1", "q3"]
        stats["limite"] = stats["q3"] + IQR_OUTLIER_FACTOR * (stats["q3"] - stats["q1"])
        out = out.merge(stats["limite"], left_on="sexo", right_index=True, how="left")
        m = (
            out["limite"].notna()
            & (out["precio_final"] > out["limite"])
            & ~out["es_outlier"]
        )
        out.loc[m, ["es_outlier", "motivo_outlier"]] = [True, "iqr_sexo"]
        out = out.drop(columns=["limite"], errors="ignore")

    return out


def _rango_eje_precio(serie) -> list[float] | None:
    """Rango Y automatico (p2-p98 + padding) para no perder el bulk de datos."""
    v = pd.to_numeric(serie, errors="coerce").dropna()
    v = v[v > 0]
    if v.empty:
        return None
    lo, hi = float(v.quantile(0.02)), float(v.quantile(0.98))
    if hi <= lo:
        hi = lo * 1.1 + 1
    pad = (hi - lo) * 0.1
    return [max(0.0, lo - pad), hi + pad]


def build_price_summary(lotes_df=None, filtrar_outliers=True):
    df = lotes_df.copy() if lotes_df is not None else load_combined_lotes()
    if df.empty:
        raise ValueError("No hay datos de lotes")
    df = marcar_outliers_lotes(df)
    CACHE["outliers_excluidos"] = df[df["es_outlier"]].copy()
    n_exc = int(df["es_outlier"].sum())
    if filtrar_outliers:
        df = df[~df["es_outlier"]].copy()
    df["fecha_subasta"] = pd.to_datetime(df["fecha_subasta"], errors="coerce")
    df["cantidad_w"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(1).clip(lower=1)
    df["precio_x_cant"] = df["precio_final"] * df["cantidad_w"]
    df["anio"] = df["fecha_subasta"].dt.year
    df["mes"] = df["fecha_subasta"].dt.month
    df["mes_nombre"] = df["mes"].map(MESES)
    summary = (
        df.groupby(["sexo", "anio", "mes", "mes_nombre"], as_index=False)
        .agg(
            precio_x_cant=("precio_x_cant", "sum"),
            animales=("cantidad_w", "sum"),
            lotes=("lote", "count"),
            precio_mediana=("precio_final", "median"),
        )
    )
    summary["precio_promedio"] = summary["precio_x_cant"] / summary["animales"]
    summary = summary.drop(columns=["precio_x_cant"]).sort_values(["sexo", "anio", "mes"])
    summary["periodo"] = summary["anio"].astype(str) + "-" + summary["mes"].astype(str).str.zfill(2)
    summary["fecha_periodo"] = pd.to_datetime(summary["periodo"] + "-01")
    summary["pct_mes_anterior"] = summary.groupby("sexo")["precio_promedio"].pct_change() * 100
    summary["pct_mismo_mes_anio_anterior"] = summary.groupby(["sexo", "mes"])["precio_promedio"].pct_change() * 100
    summary["direccion"] = summary["pct_mes_anterior"].apply(
        lambda x: "subio" if pd.notna(x) and x > 0 else ("bajo" if pd.notna(x) and x < 0 else "sin dato")
    )
    summary.attrs["outliers_excluidos"] = n_exc
    return summary


def listar_lotes_excluidos():
    """Lotes filtrados del grafico (outliers algoritmicos)."""
    return CACHE.get("outliers_excluidos", pd.DataFrame())


def build_price_by_feria(lotes_df=None, filtrar_outliers=True):
    """Precio ponderado por feria (cada punto = una subasta)."""
    df = lotes_df.copy() if lotes_df is not None else load_combined_lotes()
    df = marcar_outliers_lotes(df)
    CACHE["outliers_excluidos"] = df[df["es_outlier"]].copy()
    if filtrar_outliers:
        df = df[~df["es_outlier"]]
    df["fecha_subasta"] = pd.to_datetime(df["fecha_subasta"], errors="coerce")
    df["precio_final"] = pd.to_numeric(df["precio_final"], errors="coerce")
    df["cantidad_w"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(1).clip(lower=1)
    df["precio_x_cant"] = df["precio_final"] * df["cantidad_w"]
    out = (
        df.groupby(["sexo", "feria_no", "fecha_subasta", "ciudad"], as_index=False)
        .agg(
            precio_x_cant=("precio_x_cant", "sum"),
            animales=("cantidad_w", "sum"),
            lotes=("lote", "count"),
        )
        .sort_values(["sexo", "fecha_subasta"])
    )
    out["precio_promedio"] = out["precio_x_cant"] / out["animales"]
    return out.drop(columns=["precio_x_cant"])


def listar_sexos():
    lotes = load_combined_lotes()
    if lotes.empty or "sexo" not in lotes.columns:
        return []
    return sorted(lotes["sexo"].dropna().unique().tolist())


def _pct_feria_anterior(fer_df):
    """Variacion % vs feria anterior (mismo sexo)."""
    out = fer_df.sort_values("fecha_subasta").copy()
    out["pct_feria_anterior"] = out["precio_promedio"].pct_change() * 100
    out["label_feria"] = out.apply(lambda r: f"F{int(r['feria_no'])}", axis=1)
    return out


def plot_precio_tiempo(sexo: str):
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    fer = _pct_feria_anterior(build_price_by_feria())
    sub = fer[fer["sexo"] == sexo]
    if sub.empty:
        raise ValueError(f"Sin datos para sexo {sexo}")
    colors = ["#2ca02c" if (v or 0) >= 0 else "#d62728" for v in sub["pct_feria_anterior"].fillna(0)]
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Precio por feria {sexo}", f"Variacion % {sexo}"))
    fig.add_trace(
        go.Scatter(
            x=sub["fecha_subasta"], y=sub["precio_promedio"],
            mode="lines+markers",
            name="Por feria",
            line=dict(color="#1f77b4", width=2),
            marker=dict(size=8),
            customdata=sub[["feria_no", "animales", "lotes"]].values,
            hovertemplate=(
                "Feria: %{customdata[0]}<br>Fecha: %{x|%Y-%m-%d}<br>"
                "Precio: $%{y:,.0f}<br>Animales: %{customdata[1]}<extra></extra>"
            ),
        ),
        row=1, col=1,
    )
    fig.add_trace(go.Bar(x=sub["label_feria"], y=sub["pct_feria_anterior"], marker_color=colors, name="% cambio"), row=1, col=2)
    fig.add_hline(y=0, line_dash="dash", row=1, col=2)
    layout = dict(height=450, template="plotly_white", title=f"Precio por feria: {sexo}")
    yr = _rango_eje_precio(sub["precio_promedio"])
    if yr:
        layout["yaxis"] = dict(range=yr, title="Precio ($)")
    fig.update_layout(**layout)
    return fig


def plot_mismo_mes_anios(sexo: str):
    import plotly.graph_objects as go
    summary = build_price_summary()
    sub = summary[summary["sexo"] == sexo]
    if sub.empty:
        raise ValueError(f"Sin datos para sexo {sexo}")
    fig = go.Figure()
    for year in sorted(sub["anio"].unique()):
        ysub = sub[sub["anio"] == year]
        fig.add_trace(go.Bar(x=ysub["mes_nombre"], y=ysub["precio_promedio"], name=str(int(year))))
    layout = dict(barmode="group", title=f"Mismo mes entre anos - {sexo} (sin outliers)", template="plotly_white", height=450)
    yr = _rango_eje_precio(sub["precio_promedio"])
    if yr:
        layout["yaxis"] = dict(range=yr)
    fig.update_layout(**layout)
    return fig


def plot_precio_tiempo_selector():
    """Precio en el tiempo: cada punto = una feria/subasta (fecha real)."""
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    ferias = _pct_feria_anterior(build_price_by_feria())
    sexos = sorted(ferias["sexo"].dropna().unique().tolist())
    if not sexos:
        raise ValueError("Sin datos")
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Precio por fecha de subasta", "Variacion % vs feria anterior"),
    )
    buttons = []
    rangos = {}
    _hover = (
        "Feria: %{customdata[0]}<br>Fecha subasta: %{x|%Y-%m-%d}<br>"
        "Precio: $%{y:,.0f}<br>Animales: %{customdata[1]}<br>Lotes: %{customdata[2]}<extra></extra>"
    )
    for idx, sexo in enumerate(sexos):
        fer = ferias[ferias["sexo"] == sexo]
        colors = ["#2ca02c" if (v or 0) >= 0 else "#d62728" for v in fer["pct_feria_anterior"].fillna(0)]
        fig.add_trace(
            go.Scatter(
                x=fer["fecha_subasta"], y=fer["precio_promedio"],
                mode="lines+markers", name="Por feria",
                line=dict(color="#1f77b4", width=2),
                marker=dict(size=8),
                customdata=fer[["feria_no", "animales", "lotes"]].values,
                hovertemplate=_hover,
            ),
            row=1, col=1,
        )
        fig.add_trace(
            go.Bar(x=fer["label_feria"], y=fer["pct_feria_anterior"], marker_color=colors, name="% cambio"),
            row=1, col=2,
        )
        visible = [False] * (2 * len(sexos))
        visible[2 * idx] = visible[2 * idx + 1] = True
        rangos[sexo] = _rango_eje_precio(fer["precio_promedio"])
        layout_up = {"title": f"Precio {sexo} — un punto por feria/subasta"}
        if rangos[sexo]:
            layout_up["yaxis"] = {"range": rangos[sexo]}
        buttons.append(dict(label=str(sexo), method="update", args=[{"visible": visible}, layout_up]))
    fig.data[0].visible = fig.data[1].visible = True
    for i in range(2, len(fig.data)):
        fig.data[i].visible = False
    fig.add_hline(y=0, line_dash="dash", row=1, col=2)
    exc = CACHE.get("outliers_excluidos")
    n_exc = len(exc) if isinstance(exc, pd.DataFrame) else 0
    layout = dict(
        height=480,
        template="plotly_white",
        title=f"Precio {sexos[0]} — un punto por feria/subasta",
        showlegend=False,
        updatemenus=[dict(
            active=0, buttons=buttons, direction="down",
            showactive=True, x=0.0, xanchor="left", y=1.18, yanchor="top",
        )],
    )
    if rangos.get(sexos[0]):
        layout["yaxis"] = dict(range=rangos[sexos[0]], title="Precio ($)")
    if n_exc:
        layout["annotations"] = [dict(
            text=f"{n_exc} lotes atipicos excluidos | Cada punto = fecha real de subasta",
            xref="paper", yref="paper", x=0, y=1.28, showarrow=False, font=dict(size=11, color="#666"),
        )]
    fig.update_layout(**layout)
    return fig


def plot_mismo_mes_anios_selector():
    """Mismo mes entre anos con selector de sexo integrado en Plotly."""
    import plotly.graph_objects as go
    summary = build_price_summary()
    sexos = sorted(summary["sexo"].dropna().unique().tolist())
    if not sexos:
        raise ValueError("Sin datos")
    fig = go.Figure()
    ranges = []
    rangos = {}
    for sexo in sexos:
        start = len(fig.data)
        sub = summary[summary["sexo"] == sexo]
        rangos[sexo] = _rango_eje_precio(sub["precio_promedio"])
        for year in sorted(sub["anio"].unique()):
            ysub = sub[sub["anio"] == year]
            fig.add_trace(go.Bar(x=ysub["mes_nombre"], y=ysub["precio_promedio"], name=str(int(year))))
        ranges.append((start, len(fig.data)))
    n = len(fig.data)
    buttons = []
    for idx, sexo in enumerate(sexos):
        visible = [False] * n
        for i in range(ranges[idx][0], ranges[idx][1]):
            visible[i] = True
        layout_up = {"title": f"Mismo mes entre anos - {sexo} (escala algoritmica)"}
        if rangos[sexo]:
            layout_up["yaxis"] = {"range": rangos[sexo]}
        buttons.append(dict(label=str(sexo), method="update", args=[{"visible": visible}, layout_up]))
    init = [False] * n
    for i in range(ranges[0][0], ranges[0][1]):
        init[i] = True
    for i, trace in enumerate(fig.data):
        trace.visible = init[i]
    layout = dict(
        barmode="group",
        height=450,
        template="plotly_white",
        title=f"Mismo mes entre anos - {sexos[0]} (escala algoritmica)",
        updatemenus=[dict(
            active=0, buttons=buttons, direction="down",
            showactive=True, x=0.0, xanchor="left", y=1.18, yanchor="top",
        )],
    )
    if rangos.get(sexos[0]):
        layout["yaxis"] = dict(range=rangos[sexos[0]])
    fig.update_layout(**layout)
    return fig


def plot_bigotes_precio():
    """Diagrama de bigotes por sexo + panel de lotes atipicos excluidos."""
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    lotes = marcar_outliers_lotes(load_combined_lotes().copy())
    clean = lotes[~lotes["es_outlier"]]
    exc = lotes[lotes["es_outlier"]]
    CACHE["outliers_excluidos"] = exc.copy()

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "Ganado corriente (sin atipicos)",
            f"Atipicos excluidos ({len(exc)} lotes)",
        ),
    )

    for sexo in sorted(clean["sexo"].dropna().unique()):
        sub = clean[clean["sexo"] == sexo]
        fig.add_trace(
            go.Box(
                y=sub["precio_final"],
                name=str(sexo),
                boxpoints="outliers",
                marker_color="#1f77b4",
                showlegend=False,
            ),
            row=1, col=1,
        )

    if exc.empty:
        fig.add_annotation(
            text="Sin atipicos detectados",
            xref="x2", yref="y2", x=0.5, y=0.5, showarrow=False,
        )
    else:
        for sexo in sorted(exc["sexo"].dropna().unique()):
            sub = exc[exc["sexo"] == sexo]
            fig.add_trace(
                go.Box(
                    y=sub["precio_final"],
                    name=str(sexo),
                    boxpoints="all",
                    jitter=0.3,
                    pointpos=0,
                    marker=dict(color="#d62728", size=6),
                    line=dict(color="#d62728"),
                    showlegend=False,
                ),
                row=1, col=2,
            )

    yr = _rango_eje_precio(clean["precio_final"])
    layout = dict(
        height=520,
        template="plotly_white",
        title="Precio final por sexo — diagrama de bigotes",
        boxmode="group",
    )
    if yr:
        layout["yaxis"] = dict(title="Precio final ($)", range=yr)
    else:
        layout["yaxis"] = dict(title="Precio final ($)")
    layout["yaxis2"] = dict(title="Precio final ($)", type="log")
    fig.update_layout(**layout)
    fig.update_xaxes(tickangle=-45, row=1, col=1)
    fig.update_xaxes(tickangle=-45, row=1, col=2)
    return fig


def plot_bigotes_sexo(sexo: str):
    """Bigotes de un solo sexo: corriente vs atipicos superpuestos."""
    import plotly.graph_objects as go

    lotes = marcar_outliers_lotes(load_combined_lotes().copy())
    clean = lotes[(lotes["sexo"] == sexo) & ~lotes["es_outlier"]]
    exc = lotes[(lotes["sexo"] == sexo) & lotes["es_outlier"]]
    if clean.empty and exc.empty:
        raise ValueError(f"Sin datos para sexo {sexo}")

    fig = go.Figure()
    if not clean.empty:
        fig.add_trace(go.Box(
            y=clean["precio_final"],
            name=f"{sexo} corriente",
            boxpoints="outliers",
            marker_color="#1f77b4",
        ))
    if not exc.empty:
        fig.add_trace(go.Scatter(
            x=[sexo] * len(exc),
            y=exc["precio_final"],
            mode="markers",
            name=f"{sexo} atipicos ({len(exc)})",
            marker=dict(color="#d62728", size=10, symbol="x"),
            text=exc.apply(
                lambda r: f"Feria {r['feria_no']} Lote {r['lote']}<br>{r.get('motivo_outlier','')}",
                axis=1,
            ),
            hoverinfo="text+y",
        ))
    yr = _rango_eje_precio(clean["precio_final"]) if not clean.empty else None
    layout = dict(
        title=f"Bigotes {sexo} — azul=corriente, X rojas=atipicos excluidos",
        template="plotly_white",
        height=450,
        yaxis=dict(title="Precio final ($)", range=yr if yr else None),
    )
    fig.update_layout(**layout)
    return fig


def show_plotly(fig):
    """Renderiza Plotly en Colab via HTML (funciona fuera de widgets.Output)."""
    from IPython.display import HTML, display
    display(HTML(fig.to_html(include_plotlyjs="cdn")))


def export_csv(include_lotes=True, include_resumen=True):
    if not include_lotes and not include_resumen:
        raise ValueError("Selecciona al menos lotes o resumen")
    if include_lotes:
        lotes_path = COMBINED_DIR / "lotes.parquet"
        if not lotes_path.exists():
            raise FileNotFoundError("No hay combined/lotes.parquet. Descarga ferias primero.")
        lotes = pd.read_parquet(lotes_path)
        lotes.to_csv(EXPORT_DIR / "lotes_consolidado.csv", index=False, encoding="utf-8-sig")
        print(f"CSV lotes: {EXPORT_DIR / 'lotes_consolidado.csv'} ({len(lotes)} filas)")
    if include_resumen:
        resumen_path = COMBINED_DIR / "resumen.parquet"
        if not resumen_path.exists():
            print("Aviso: no hay combined/resumen.parquet")
        else:
            resumen = pd.read_parquet(resumen_path)
            resumen.to_csv(EXPORT_DIR / "resumen_consolidado.csv", index=False, encoding="utf-8-sig")
            print(f"CSV resumen: {EXPORT_DIR / 'resumen_consolidado.csv'} ({len(resumen)} filas)")


# --- ML: categorias de compra y EDA ---
import re as _re_ml

SEXO_A_CATEGORIA_ML = {
    "ML": "novillo_flaco", "NC": "novillo_flaco", "BF": "novillo_flaco", "NG": "novillo_flaco",
    "VC": "vaca_flaca", "HL": "vaca_flaca",
    "VP": "vaca_parida",
    "MG": None, "MC": None, "VG": None, "EQ": None, "XX": None,
}
CATEGORIAS_ML = ["novillo_flaco", "vaca_flaca", "vaca_parida"]
ML_MIN_LOTES, ML_MIN_FERIAS = 500, 12
ML_PESO_MAX_NOVILLO, ML_PESO_MAX_VACA = 380, 420
_MIXED_OBS = _re_ml.compile(r"\d+\s*[VHMCNG]|/\s*\d|Y\s+\d+\s+[VHMCNG]|MEZCL", _re_ml.IGNORECASE)


def asignar_categoria_ml(sexo, peso_promedio=None):
    cat = SEXO_A_CATEGORIA_ML.get(str(sexo).strip().upper())
    if cat is None or peso_promedio is None or peso_promedio <= 0:
        return cat
    if cat == "novillo_flaco" and peso_promedio > ML_PESO_MAX_NOVILLO:
        return None
    if cat == "vaca_flaca" and peso_promedio > ML_PESO_MAX_VACA:
        return None
    return cat


def preparar_lotes_ml(df=None):
    out = (load_combined_lotes() if df is None else df).copy()
    out["fecha_subasta"] = pd.to_datetime(out["fecha_subasta"], errors="coerce")
    out["sexo"] = out["sexo"].astype(str).str.strip().str.upper()
    out["peso_promedio"] = pd.to_numeric(out["peso_promedio"], errors="coerce")
    out["precio_final"] = pd.to_numeric(out["precio_final"], errors="coerce")
    out["precio_base"] = pd.to_numeric(out["precio_base"], errors="coerce")
    out["cantidad"] = pd.to_numeric(out["cantidad"], errors="coerce")
    out["precio_kg"] = pd.to_numeric(
        out["precio_final"] / out["peso_promedio"].replace(0, pd.NA), errors="coerce"
    )
    out["categoria_ml"] = [
        asignar_categoria_ml(s, p) for s, p in zip(out["sexo"], out["peso_promedio"], strict=True)
    ]
    obs = out["observaciones"].fillna("").astype(str)
    out["lote_mixto"] = obs.str.contains(_MIXED_OBS, regex=True)
    out["mes"] = out["fecha_subasta"].dt.month
    return out


def reporte_ml_eda(df=None):
    """Analisis exploratorio para ML (precio maximo de compra)."""
    from IPython.display import display
    if not datos_precargados():
        precargar_datos(rebuild=False, verbose=True)
    df = preparar_lotes_ml(df)
    if df.empty:
        print("No hay lotes. Descarga ferias en Celda B.")
        return df

    def _sep(t):
        print("\n" + "=" * 60 + "\n" + t + "\n" + "=" * 60)

    _sep("1. VOLUMEN Y COBERTURA")
    print(f"  Lotes totales     : {len(df):,}")
    print(f"  Ferias            : {df['feria_no'].nunique()} ({df['feria_no'].min()} - {df['feria_no'].max()})")
    fechas = df["fecha_subasta"].dropna()
    if not fechas.empty:
        print(f"  Rango fechas      : {fechas.min().date()} -> {fechas.max().date()}")
    print(f"  Ciudades          : {', '.join(sorted(df['ciudad'].dropna().unique()))}")

    _sep("2. CALIDAD DE DATOS")
    checks = {
        "precio_final nulo/cero": (df["precio_final"].isna() | (df["precio_final"] <= 0)).sum(),
        "peso_promedio nulo/cero": (df["peso_promedio"].isna() | (df["peso_promedio"] <= 0)).sum(),
        "fecha_subasta nula": df["fecha_subasta"].isna().sum(),
        "sexo desconocido": (~df["sexo"].isin(SEXO_A_CATEGORIA_ML.keys())).sum(),
        "lotes mixtos (obs)": int(df["lote_mixto"].sum()),
        "precio_kg outlier >50k": int((df["precio_kg"] > 50_000).sum()),
    }
    for k, v in checks.items():
        pct = 100 * v / len(df) if len(df) else 0
        print(f"  {k:28}: {v:5} ({pct:4.1f}%) {'!' if v else 'OK'}")

    _sep("3. DISTRIBUCION POR CODIGO SEXO")
    g = (
        df.groupby("sexo", as_index=False)
        .agg(
            lotes=("lote", "count"),
            peso_med=("peso_promedio", "median"),
            precio_med=("precio_final", "median"),
            precio_kg_med=("precio_kg", "median"),
            categoria=("categoria_ml", lambda x: x.dropna().iloc[0] if x.dropna().any() else "-"),
        )
        .sort_values("lotes", ascending=False)
    )
    display(g)

    _sep("4. CATEGORIAS ML (novillo flaco / vaca flaca / vaca parida)")
    cat = df[df["categoria_ml"].notna()]
    print(f"  Lotes asignados  : {len(cat):,} ({100 * len(cat) / len(df):.1f}%)")
    print(f"  Excluidos        : {len(df) - len(cat):,}\n")
    for c in CATEGORIAS_ML:
        sub = cat[cat["categoria_ml"] == c]
        n, ferias = len(sub), sub["feria_no"].nunique()
        ok_n = "OK" if n >= ML_MIN_LOTES else "NO"
        ok_f = "OK" if ferias >= ML_MIN_FERIAS else "NO"
        print(f"  {c:16} | lotes={n:5} [{ok_n} min {ML_MIN_LOTES}] | ferias={ferias:3} [{ok_f} min {ML_MIN_FERIAS}]")
        if n:
            print(f"                     peso med={sub['peso_promedio'].median():.0f} kg | "
                  f"precio/kg med=${sub['precio_kg'].median():,.0f} | "
                  f"precio/cab med=${sub['precio_final'].median():,.0f}")

    _sep("5. PRECIOS POR CATEGORIA Y MES ($/kg)")
    if not cat.empty:
        pivot = cat.groupby(["categoria_ml", "mes"])["precio_kg"].agg(["count", "median", "std"]).round(0).reset_index()
        for c in CATEGORIAS_ML:
            sub = pivot[pivot["categoria_ml"] == c]
            if not sub.empty:
                print(f"\n  --- {c} ---")
                display(sub.drop(columns="categoria_ml"))

    _sep("6. CORRELACIONES vs precio_kg")
    num = cat[["peso_promedio", "precio_base", "precio_final", "precio_kg", "cantidad"]].dropna()
    if len(num) > 10:
        corr = num.corr(numeric_only=True)
        if "precio_kg" in corr.columns:
            for col, val in corr["precio_kg"].drop("precio_kg").sort_values(key=abs, ascending=False).items():
                print(f"  {col:16}: {val:+.3f}")

    _sep("7. LISTO PARA ML?")
    for c in CATEGORIAS_ML:
        sub = cat[cat["categoria_ml"] == c]
        n, f = len(sub), sub["feria_no"].nunique()
        if n >= ML_MIN_LOTES and f >= ML_MIN_FERIAS:
            print(f"  {c}: SI - entrenar modelo completo")
        elif n >= 100:
            print(f"  {c}: PARCIAL - baseline + modelo simple (lotes={n}, ferias={f})")
        else:
            print(f"  {c}: NO AUN - solo tablas (lotes={n}, ferias={f})")
    ferias_tot = df["feria_no"].nunique()
    if ferias_tot < ML_MIN_FERIAS:
        print(f"\n  -> Descarga mas ferias (tienes {ferias_tot}, ideal >={ML_MIN_FERIAS}).")
    print("  -> Target recomendado: precio_kg. Excluir lotes mixtos al entrenar.")
    return df


ultima = ultima_feria_en_drive()
total = len(listar_ferias_en_drive())
print(f"Listo. Ferias en Drive: {total} | Ultima: {ultima}")
print("Ahora ejecuta la celda de interfaz.")

In [ ]:
# Celda B - Panel interactivo (Descarga · Datos · Exportar)
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

_OUT_STYLE = {"border": "1px solid #ccc", "padding": "8px", "max_height": "360px", "overflow": "auto"}


def _ferias_options():
    f = listar_ferias_en_drive()
    return f if f else [0]


def _stats_html(s):
    return (
        f"<b>Ferias:</b> {s['ferias']} ({s['primera']} → {s['ultima']}) &nbsp;|&nbsp; "
        f"<b>Lotes:</b> {s['lotes_total']:,} &nbsp;|&nbsp; "
        f"<b>Fechas:</b> {s['fecha_min'] or '—'} → {s['fecha_max'] or '—'}<br>"
        f"<b>Ciudades:</b> {', '.join(s['ciudades'][:8])}{'…' if len(s['ciudades']) > 8 else ''}"
    )


_ultima = ultima_feria_en_drive()
_total = len(listar_ferias_en_drive())

ui_estado = widgets.HTML(value="<span style='color:#888'>⏳ Precargando datos...</span>")
btn_precargar = widgets.Button(description="Precargar / refrescar datos", button_style="warning", icon="folder-open")
ui_log_precarga = widgets.Output(layout={"max_height": "72px", "overflow": "auto", "font-size": "12px"})

ui_titulo = widgets.HTML(
    value=(
        "<h3 style='margin-bottom:4px'>Subastas Casanare</h3>"
        f"<p style='color:#555'>Drive: <code>subasta_ganadera_data</code> &nbsp;|&nbsp; "
        f"Ferias: <b>{_total}</b> &nbsp;|&nbsp; Ultima: <b>{_ultima or '—'}</b></p>"
        "<p><small>Graficos → ejecuta <b>Celda C</b></small></p>"
    )
)

ui_desde = widgets.IntText(value=_ultima + 1 if _ultima else 1950, description="Desde:", style={"description_width": "55px"}, layout={"width": "130px"})
ui_hasta = widgets.IntText(value=(_ultima + 50) if _ultima else 2000, description="Hasta:", style={"description_width": "55px"}, layout={"width": "130px"})
ui_force = widgets.Checkbox(value=False, description="Reprocesar existentes")
ui_feria_una = widgets.IntText(value=_ultima + 1 if _ultima else 1950, description="Feria:", style={"description_width": "55px"}, layout={"width": "130px"})
ui_progress = widgets.IntProgress(value=0, min=0, max=100, description="Progreso:", bar_style="info", layout={"width": "98%"})
ui_log_desc = widgets.Output(layout=_OUT_STYLE)

btn_detectar = widgets.Button(description="Detectar ultima", button_style="info", icon="search")
btn_descargar = widgets.Button(description="Descargar rango", button_style="success", icon="download")
btn_una = widgets.Button(description="Descargar 1 feria", icon="cloud-download")
btn_rebuild = widgets.Button(description="Regenerar consolidado", icon="refresh")

ui_feria_ver = widgets.Dropdown(options=_ferias_options(), description="Feria:", style={"description_width": "55px"})
ui_stats = widgets.HTML(value="<i>Los datos se cargan al iniciar la celda.</i>")
ui_log_datos = widgets.Output(layout=_OUT_STYLE)

btn_ver_lotes = widgets.Button(description="Ver lotes", icon="table")
btn_stats = widgets.Button(description="Resumen general", button_style="info", icon="info")
btn_refresh_ferias = widgets.Button(description="Actualizar lista", icon="refresh")

ui_exp_lotes = widgets.Checkbox(value=True, description="CSV lotes")
ui_exp_resumen = widgets.Checkbox(value=True, description="CSV resumen")
ui_log_exp = widgets.Output(layout=_OUT_STYLE)

btn_csv_drive = widgets.Button(description="Guardar CSV en Drive", button_style="success", icon="save")
btn_dl_lotes = widgets.Button(description="Descargar lotes al PC", icon="download")
btn_dl_resumen = widgets.Button(description="Descargar resumen al PC", icon="download")


def _aplicar_precarga(result):
    if result["ok"]:
        ui_estado.value = f"<span style='color:#2e7d32'>✓ {result['mensaje']}</span>"
        ui_stats.value = _stats_html(result["stats"])
    else:
        ui_estado.value = f"<span style='color:#e65100'>⚠ {result['mensaje']}</span>"
        ui_stats.value = "<i>Sin datos. Descarga ferias en la pestaña Descarga.</i>"
    _actualizar_estado()


def on_precargar(_=None):
    btn_precargar.disabled = True
    ui_estado.value = "<span style='color:#888'>⏳ Leyendo Parquet desde Drive...</span>"
    with ui_log_precarga:
        clear_output(wait=True)
        try:
            result = precargar_datos(rebuild=False, verbose=True)
            _aplicar_precarga(result)
        except Exception as e:
            ui_estado.value = f"<span style='color:red'>Error: {e}</span>"
            print(f"Error: {e}")
    btn_precargar.disabled = False


def _actualizar_estado():
    u = ultima_feria_en_drive()
    t = len(listar_ferias_en_drive())
    ui_titulo.value = (
        "<h3 style='margin-bottom:4px'>Subastas Casanare</h3>"
        f"<p style='color:#555'>Drive: <code>subasta_ganadera_data</code> &nbsp;|&nbsp; "
        f"Ferias: <b>{t}</b> &nbsp;|&nbsp; Ultima: <b>{u or '—'}</b></p>"
        "<p><small>Graficos → ejecuta <b>Celda C</b></small></p>"
    )
    opts = _ferias_options()
    ui_feria_ver.options = opts
    if opts and opts != [0]:
        ui_feria_ver.value = opts[-1]


def _set_busy(busy: bool):
    for b in [btn_detectar, btn_descargar, btn_una, btn_rebuild, btn_csv_drive, btn_dl_lotes, btn_dl_resumen]:
        b.disabled = busy


def on_detectar(_):
    u = ultima_feria_en_drive()
    ui_desde.value = u + 1 if u else 1950
    ui_hasta.value = (u + 50) if u else 2000
    ui_feria_una.value = u + 1 if u else 1950
    _actualizar_estado()


def on_descargar(_):
    _set_busy(True)
    ui_progress.value = 0
    with ui_log_desc:
        clear_output(wait=True)
        print(f"Descargando {ui_desde.value} → {ui_hasta.value} ...\n")

        def prog(pct, feria):
            ui_progress.value = pct
            if feria:
                ui_progress.description = f"Feria {feria}:"

        try:
            run_batch(ui_desde.value, ui_hasta.value, force=ui_force.value, progress_cb=prog)
            on_precargar()
        except Exception as e:
            print(f"Error: {e}")
    ui_progress.description = "Progreso:"
    _set_busy(False)


def on_una(_):
    _set_busy(True)
    with ui_log_desc:
        clear_output(wait=True)
        try:
            run_single_feria(ui_feria_una.value, force=ui_force.value)
            on_precargar()
        except Exception as e:
            print(f"Error: {e}")
    _set_busy(False)


def on_rebuild(_):
    with ui_log_desc:
        clear_output(wait=True)
        try:
            lotes, resumen = rebuild_combined()
            print(f"Consolidado: {len(lotes)} lotes | {len(resumen)} resumen")
            on_precargar()
        except Exception as e:
            print(f"Error: {e}")


def on_ver_lotes(_):
    with ui_log_datos:
        clear_output(wait=True)
        try:
            df = load_feria_lotes(ui_feria_ver.value)
            print(f"Feria {ui_feria_ver.value}: {len(df)} lotes\n")
            display(df.head(20))
        except Exception as e:
            print(f"Error: {e}")


def on_stats(_):
    try:
        if not datos_precargados():
            on_precargar()
            return
        s = stats_drive()
        ui_stats.value = _stats_html(s)
        _actualizar_estado()
    except Exception as e:
        ui_stats.value = f"<span style='color:red'>Error: {e}</span>"


def on_refresh_ferias(_):
    on_precargar()


def on_csv_drive(_):
    _set_busy(True)
    with ui_log_exp:
        clear_output(wait=True)
        try:
            export_csv(include_lotes=ui_exp_lotes.value, include_resumen=ui_exp_resumen.value)
        except Exception as e:
            print(f"Error: {e}")
    _set_busy(False)


def on_dl_lotes(_):
    path = EXPORT_DIR / "lotes_consolidado.csv"
    if not path.exists():
        with ui_log_exp:
            clear_output(wait=True)
            print("Primero exporta CSV a Drive (o no existe el archivo).")
        return
    files.download(str(path))


def on_dl_resumen(_):
    path = EXPORT_DIR / "resumen_consolidado.csv"
    if not path.exists():
        with ui_log_exp:
            clear_output(wait=True)
            print("Primero exporta CSV resumen a Drive.")
        return
    files.download(str(path))


btn_precargar.on_click(on_precargar)
btn_detectar.on_click(on_detectar)
btn_descargar.on_click(on_descargar)
btn_una.on_click(on_una)
btn_rebuild.on_click(on_rebuild)
btn_ver_lotes.on_click(on_ver_lotes)
btn_stats.on_click(on_stats)
btn_refresh_ferias.on_click(on_refresh_ferias)
btn_csv_drive.on_click(on_csv_drive)
btn_dl_lotes.on_click(on_dl_lotes)
btn_dl_resumen.on_click(on_dl_resumen)

tab_descarga = widgets.VBox([
    widgets.HTML("<b>Descarga masiva</b>"),
    widgets.HBox([ui_desde, ui_hasta, ui_force]),
    widgets.HBox([btn_detectar, btn_descargar]),
    widgets.HTML("<b>Una feria</b>"),
    widgets.HBox([ui_feria_una, btn_una, btn_rebuild]),
    ui_progress,
    widgets.HTML("<b>Log:</b>"),
    ui_log_desc,
])

tab_datos = widgets.VBox([
    widgets.HBox([ui_feria_ver, btn_ver_lotes, btn_refresh_ferias]),
    widgets.HBox([btn_stats]),
    ui_stats,
    widgets.HTML("<b>Vista previa:</b>"),
    ui_log_datos,
])

tab_exportar = widgets.VBox([
    widgets.HBox([ui_exp_lotes, ui_exp_resumen]),
    widgets.HBox([btn_csv_drive, btn_dl_lotes, btn_dl_resumen]),
    widgets.HTML("<b>Log:</b>"),
    ui_log_exp,
])

tabs = widgets.Tab(children=[tab_descarga, tab_datos, tab_exportar])
tabs.set_title(0, "Descarga")
tabs.set_title(1, "Datos")
tabs.set_title(2, "Exportar")

display(widgets.VBox([
    ui_titulo,
    widgets.HBox([btn_precargar, ui_estado]),
    ui_log_precarga,
    tabs,
]))

on_precargar()


## Celda C — Graficos

Ejecuta **despues de Celda A**. Plotly con selector de sexo; escala algoritmica excluye atipicos.

- **C1** — precio en el tiempo + mismo mes entre anos
- **C2** — diagrama de bigotes (corriente vs atipicos)
- **C3** — tabla analitica (formulario Colab)


In [ ]:
# Celda C1 - Graficos (escala algoritmica + selector Plotly)
from IPython.display import HTML, display

if not datos_precargados():
    precargar_datos(rebuild=False, verbose=True)

if not listar_sexos():
    print("No hay datos. Usa Celda B → Descarga ferias, luego vuelve aqui.")
else:
    build_price_summary()  # refresca summary y outliers en CACHE
    exc = listar_lotes_excluidos()
    if not exc.empty:
        print(f"Escala algoritmica: {len(exc)} lotes atipicos excluidos del grafico.")
        print("Ver detalle: listar_lotes_excluidos()")
    print("Linea azul: cada punto = una feria/subasta (fecha real). Pasa el mouse: feria, fecha, precio.")
    show_plotly(plot_precio_tiempo_selector())
    show_plotly(plot_mismo_mes_anios_selector())


### C2 — Diagrama de bigotes

Izquierda: precios **corrientes** por sexo. Derecha: **atipicos excluidos** (escala log). Detalle: `listar_lotes_excluidos()`.


In [ ]:
# Celda C2 - Diagrama de bigotes (corriente vs atipicos)
if not datos_precargados():
    precargar_datos(rebuild=False, verbose=True)

if not listar_sexos():
    print("No hay datos.")
else:
    show_plotly(plot_bigotes_precio())
    print("\nEjemplo un sexo (BF) — X rojas = atipicos:")
    show_plotly(plot_bigotes_sexo("BF"))


In [ ]:
#@title Tabla analitica { run: "auto" }
#@markdown Cambia **Sexo** abajo (ej. BF, ML, VG). La celda se re-ejecuta sola.

if not datos_precargados():
    precargar_datos(rebuild=False, verbose=False)

_sexos = listar_sexos()
sexo = _sexos[0] if _sexos else "BF"  # @param {type:"string"}

if not _sexos:
    print("No hay datos.")
elif sexo not in _sexos:
    print(f"Sexo '{sexo}' no existe. Opciones: {', '.join(_sexos)}")
else:
    summary = CACHE.get("summary")
    if summary is None:
        summary = build_price_summary()
    sub = summary[summary["sexo"] == sexo]
    cols = ["periodo", "precio_promedio", "pct_mes_anterior", "direccion", "pct_mismo_mes_anio_anterior"]
    display(sub[cols].round(1).tail(24))


## Celda D — Analisis ML (EDA)

Ejecuta **despues de Celda A** (y Celda B si aun no hay datos).

Explora volumen, calidad y categorias **novillo flaco / vaca flaca / vaca parida** antes de entrenar modelos de precio maximo de compra.


In [ ]:
# Celda D - Analisis exploratorio ML
ml_df = reporte_ml_eda()
